# Importing necessary libraries

In [0]:
from pyspark.sql import Row, Window
from pyspark.sql.types import (
    StringType,
    IntegerType,
    StructType,
    DateType,
    DecimalType,
    StructField,
)
from pyspark.sql.functions import (
    when,
    col,
    sum,
    row_number,
    date_format,
    round,
    avg,
    countDistinct,
    desc,
    max,
    concat,
    lit,
    month,
    to_date,
    year,
    ntile,
)
import plotly.express as px
from delta.tables import DeltaTable

# Phase 1 — Setup & Data Loading 

## 4. Read the CSV file using PySpark
## 5. Infer schema automatically


The given CSV file is loaded from the Unity Catalog volume, with `inferSchema = True` to infer schema automatically.

In [0]:
df = spark.read.csv(
    "/Volumes/supermarket_sales/supermarket_sales/data/Superstore.csv",
    inferSchema = True,
    header = True,
    quote = '"',
    escape = '"'
)

## 6. Display First 10 rows

**Description:** Displaying the first 10 rows of the loaded CSV file.\
**Function used:** `.limit()`

In [0]:
df.limit(10).display()

## 7. Check total number of rows

**Description:** Displaying the total number of rows on the DataFrame\
**Function used:** `.count()`

In [0]:
print(f"Total number of rows: {df.count()}")

## 8. Total number of columns

**Description:** Displaying the total number of columns, by finding the length of `df.columns` attribute.\
**Function used:** `len()`

In [0]:
print(f"Total number of columns: {len(df.columns)}")

## 9. Print Dataset Schema

**Description:** Displaying the data schema of the DataFrame.\
**Function used:** `printSchema()`

In [0]:
df.printSchema()

## 10. Identify Column Data Types

**Description:** Displaying the data types of each column by looping through their schema in the DataFrame\
**Attribute used:** `.dataType`

In [0]:
for column in df.schema:
    print(f"Column name: {column.name:15} Data Type: {column.dataType}")

# Phase 2 — Data Cleaning 

## 11. Check for null values in all columns



In [0]:
for column in df.columns:
    null_count = df.select(
        sum(
            when(col(column).isNull(), 1).otherwise(0)
        ).alias("null_count")
    ).collect()[0]["null_count"]
    
    print(f"{column:15} Nulls found: {null_count}")

## 12. Find duplicate records.

When considering every column for finding duplicate values, we see that there is no duplicate records. This is because of the unique values in the `Row ID` column, which is a unique ID for each row. Thus, we get an empty DataFrame while displaying the duplicate records.

In [0]:
display(
    df.groupBy(df.columns).count().filter('count > 1')
)

If we are excluding the `Row ID` column, and considering all other columns for deduplication, we get a single row, which is a duplicate:

In [0]:
display(
    df.groupBy(df.columns[1:]).count().filter('count > 1')
)

## 13. Remove duplicate records
We remove the one duplicate record, and update the DataFrame

In [0]:
df = df.dropDuplicates(df.columns[1:])

Compare the following number of rows to the <a href = "https://dbc-224b3dfc-629d.cloud.databricks.com/editor/notebooks/1386621215397577?o=7474647936215392#command/6687490459806434"> initial number of rows </a>, we would see a row's count is less, meaning we have deleted our duplicate row!

In [0]:
df.count()

## 14. Convert Order Date to proper date format.  

In [0]:
df = df.withColumn("Order Date", date_format("Order Date", "MM-dd-yyyy")) # Reordering the date format to Month-Day-Year format

In [0]:
df.limit(10).display()

## 15. Convert Ship Date to proper date format.

In [0]:
df = df.withColumn("Ship Date", date_format("Ship Date", "MM-dd-yyyy")) # Reordering the date format to Month-Day-Year format

df.limit(10).display()

## 16. Check for invalid sales values.  

What all constitutes an invalid sales value?
1. Null values
2. Negative sales (Valid if accounting refunds)

In [0]:
display(
    df.withColumn(
        "ProperSales?", # Creating a new column to store the result
        when(
            (col("Sales").isNotNull()) & # Checking if the column is not null
            (col("Sales") >= 0.0), # Checking if the values are greater than or equal to zero, eliminating negative values
            "Yes"
        ).otherwise("No") 
    ).groupBy("ProperSales?").count() # Grouping the result column to show the count of distinct values
)

## 17. Check for negative profit values.  

In [0]:
display(
    df.withColumn(
        "ProfitValues",
        when(
            (col("Profit") > 0), "Positives"
        ).when(
            (col("Profit") < 0), "Negatives"
        ).when(
            (col("Profit") == 0), "Zeros"
        ).otherwise("Invalid")
    ).groupBy("ProfitValues").count()
)

## 18. Create a cleaned DataFrame.  

In [0]:
cleaned_df = (
    df
    .withColumn("Row ID", col("Row ID").cast("int"))
    .withColumn("Order ID", col("Order ID").cast("string"))
    .withColumn("Order Date", to_date(col("Order Date"), "MM-dd-yyyy"))
    .withColumn("Ship Date", to_date(col("Ship Date"), "MM-dd-yyyy"))
    .withColumn("Ship Mode", col("Ship Mode").cast("string"))
    .withColumn("Customer ID", col("Customer ID").cast("string"))
    .withColumn("Customer Name", col("Customer Name").cast("string"))
    .withColumn("Segment", col("Segment").cast("string"))
    .withColumn("Country", col("Country").cast("string"))
    .withColumn("City", col("City").cast("string"))
    .withColumn("State", col("State").cast("string"))
    .withColumn("Postal Code", col("Postal Code").cast("string"))
    .withColumn("Region", col("Region").cast("string"))
    .withColumn("Product ID", col("Product ID").cast("string"))
    .withColumn("Category", col("Category").cast("string"))
    .withColumn("Sub-Category", col("Sub-Category").cast("string"))
    .withColumn("Product Name", col("Product Name").cast("string"))
    .withColumn("Sales", col("Sales").cast("decimal(18,2)"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("Discount", col("Discount").cast("decimal(5,4)"))
    .withColumn("Profit", col("Profit").cast("decimal(18,4)"))
)

## 19. Rename columns into standardized format.  

In [0]:
column_names = {
    _column_: _column_.strip().replace(" ", "").replace("-", "") 
    for _column_ in cleaned_df.columns
    }

cleaned_df = cleaned_df.withColumnsRenamed(column_names)

In [0]:
cleaned_df.columns

## 20. Save cleaned data as a temporary view.  

In [0]:
cleaned_df.createOrReplaceTempView("superstore_data")

# Phase 3 — Basic Analysis

## 21. Find total sales. 

In [0]:
display(
    cleaned_df.select(
        round(
            sum("Sales"),
            2
        ).alias("TotalSales")
    )
)

## 22. Find total profit.  

In [0]:
display(
    cleaned_df.select(
        round(
            sum("Profit"),
            2
        ).alias("TotalProfit")
    )
)

## 23. Find average sales per order.  

In [0]:
display(
    cleaned_df.groupBy("OrderID")
    .agg(
        round(
            avg("Sales"),
            2
        ).alias("AvgSales")
    ).orderBy("OrderID")
)

## 24. Find total number of customers.  

In [0]:
display(
    cleaned_df.agg(
        countDistinct("CustomerID")
        .alias("NumberOfCustomers")
    )
)

## 25. Find total number of products.  

In [0]:
display(
    cleaned_df.agg(
        countDistinct("ProductID")
        .alias("NumberOfProducts")
    )
)

## 26. Find unique categories. 

In [0]:
display(
    cleaned_df.select("Category")
    .distinct()
    .orderBy("Category")
)

## 27. Find unique sub-categories.  

In [0]:
display(
    cleaned_df.select("SubCategory")
    .distinct()
    .orderBy("SubCategory")
)

## 28. Find total orders by region.  

In [0]:
display(
    cleaned_df.groupBy("Region")
    .count()
    .alias("TotalOrders")
    .orderBy("Region")
)

## 29. Find total sales by region.  

In [0]:
display(
    cleaned_df.groupBy("Region")
    .agg(
        sum("Sales").alias("TotalSales")
    ) 
    .orderBy("Region")
)

## 30. Find total profit by region.  

In [0]:
display(
    cleaned_df.groupBy("Region")
    .agg(
        round(sum("Profit"), 2).alias("TotalProfit")
    ) 
    .orderBy("Region")
)

# Phase 4 — Intermediate Analysis 

## 31. Find top 10 products by sales.  

In [0]:
display(
    cleaned_df.groupBy("ProductID")
    .agg(
        sum("Sales").alias("TotalSales")
    )
    .orderBy(desc("TotalSales"))
    .limit(10)
)

## 32. Find bottom 10 products by profit.  

In [0]:
display(
    cleaned_df.groupBy("ProductID")
    .agg(
        round(
            sum("Profit"),
            2
        ).alias("ProductProfit")
    ).orderBy(col("ProductProfit").asc())
    .limit(10)
)

## 33. Find most profitable category.  

In [0]:
display(
    cleaned_df.groupBy("Category")
    .agg(
        round(
            sum(col("Profit")),
            2
        ).alias("CategoryProfit")
    ).orderBy(desc("CategoryProfit"))
    .limit(1)
)

## 34. Find least profitable category.

In [0]:
display(
    cleaned_df.groupBy("Category")
    .agg(
        round(
            sum(col("Profit")),
            2
        ).alias("CategoryProfit")
    ).orderBy("CategoryProfit")
    .limit(1)
)

## 35. Find highest selling sub-category.  

In [0]:
display(
    cleaned_df.groupBy("SubCategory")
    .agg(
        round(
            sum(col("Sales")),
            2
        ).alias("SubCategorySales")
    ).orderBy(desc("SubCategorySales"))
    .limit(1)
)

## 36. Find average discount by category.  

In [0]:
display(
    cleaned_df.groupBy("Category")
    .agg(
        concat(
                round(
                    avg("Discount") * 100,
                    2
                ),
                lit("%")
        )
       .alias("AverageDiscount")
    ).orderBy("AverageDiscount")
)

## 37. Find monthly sales trend.  

In [0]:
display(
    cleaned_df.groupBy(
        date_format(col("OrderDate"), "MM")
        .alias("Month")
    )
    .agg(
        round(
            sum("Sales"),
            2
        ).alias("Monthly Sales")
    ).orderBy(col("Month"))
)

## 38. Find yearly sales trend.  

In [0]:
display(
    cleaned_df.groupBy(
        year(col("OrderDate"))
        .alias("Year")
    )
    .agg(
        round(
            sum("Sales"),
            2
        ).alias("Yearly Sales")
    ).orderBy(desc("Yearly Sales"))
)

## 39. Find top 5 customers by sales.  

In [0]:
display(
    cleaned_df.groupBy("CustomerID")
    .agg(
        round(
            sum(col("Sales")),
            2
        ).alias("CustomerSales")
    ).orderBy(desc("CustomerSales"))
    .limit(5)
)

## 40. Find top 5 loss-making products.  

In [0]:
display(
    cleaned_df.groupBy(
        col("ProductID")
    ).agg(
        round(
            sum(col("Profit")),
            2
        ).alias("Loss")
    ).orderBy(col("Loss").asc())
    .limit(5)
)

# Phase 5 — Advanced PySpark Tasks 

## 41. Create a new column for Profit Margin.  

In [0]:
cleaned_df = cleaned_df.withColumn(
    "ProfitMargin",
    round(
        col("Profit") / col("Sales"),
        5
    )
)

cleaned_df.limit(10).display()

## 42. Categorize orders into High/Medium/Low sales.  

In [0]:
window = Window.orderBy(col("Sales").desc())

cleaned_df = cleaned_df.withColumn(
    "SalesCategory",
    when(
        ntile(3).over(window) == 1,
        "High"
    ).when(
        ntile(3).over(window) == 2,
        "Medium"
    ).otherwise(
        "Low"
    )
)

cleaned_df.limit(10).display()

## 43. Use Window Functions for ranking products.  

In [0]:
window = Window.orderBy(col("ProductID"))

display(
    cleaned_df
)

## 44. Find cumulative sales over time.

In [0]:
window = Window.orderBy(col("OrderDate")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative_sum = cleaned_df.withColumn(
    "CumulativeSales",
    round(
        sum(col("Sales")).over(window),
        2
    )
)

cumulative_sum.limit(10).display()

## 45. Find moving average of sales.  

In [0]:
window = Window.orderBy(col("OrderDate")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

moving_avg = cleaned_df.withColumn(
    "MovingAverage",
    round(
        avg(col("Sales")).over(window),
        2
    )
)

moving_avg.limit(10).display()

## 46. Partition data by region.  

In [0]:
window = Window.partitionBy(col("Region")).orderBy(col("Region"))

display(
    cleaned_df.withColumn(
        "TotalSales",
        round(
            sum(col("Sales"))
            .over(window),
            2
        )
    )
)

## 47. Cache frequently used DataFrames.

In [0]:
%skip

cleaned_df.cache()

## 48. Explain the execution plan using `.explain()`.  

In [0]:
display(cleaned_df.explain())

## 50. Write optimized PySpark queries.  

In [0]:
superstore_data = DeltaTable.forPath(
    spark,
    path = "supermarket_sales.supermarket_sales.superstore"
)

superstore_data.optimize()

# Phase 6 — SQL Tasks in Databricks 

## 51. Create SQL tables from DataFrames.  

In [0]:
%skip

cleaned_df.write.mode("overwrite").saveAsTable("supermarket_sales.supermarket_sales.superstore_sales")

## 52. Run SQL queries for total sales.  

In [0]:
%sql
USE CATALOG supermarket_sales;
USE SCHEMA supermarket_sales;

In [0]:
%sql
SELECT
    ROUND(SUM(Sales), 2) AS TotalSales
FROM
    superstore_sales;

## 53. Run SQL queries for top customers.  

In [0]:
%sql

WITH CustomerSales AS(
    SELECT
        CustomerID,
        CustomerName,
        Sales,
        SUM(Sales) OVER (PARTITION BY CustomerID) AS TotalCustomerSales, 
        ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY Sales DESC) AS CustomerRank
    FROM superstore_sales
    GROUP BY CustomerID, CustomerName, Sales
)
SELECT
    CustomerID,
    CustomerName,
    TotalCustomerSales
FROM
    CustomerSales
WHERE CustomerRank = 1
ORDER BY TotalCustomerSales DESC
LIMIT 10;

## 54. Run SQL queries for monthly trends.  

In [0]:
%sql

WITH SalesDetails AS(
    SELECT
        MONTH(orderdate) AS SalesMonth,
        SUM(Sales) AS TotalSales
    FROM
        superstore_sales
    GROUP BY SalesMonth
)
SELECT
    SalesMonth,
    TotalSales
FROM
    SalesDetails
ORDER BY SalesMonth;

## 55. Use GROUP BY and HAVING clauses.  

In [0]:
%sql
-- Identify sub-categories with a product count between 100 and 500

SELECT
    SubCategory,
    COUNT(ProductID) AS Products
FROM
    superstore_sales
GROUP BY SubCategory
HAVING COUNT(ProductID) BETWEEN 100 AND 500;

## 56. Use JOIN operations if multiple tables are created.  

In [0]:
%skip
-- Use JOIN operations if multiple tables are created
SELECT
    c.customer_name,
    c.segment,
    p.category,
    ROUND(SUM(m.sales), 2) as total_sales
FROM superstore_main m
INNER JOIN dim_customers c ON m.customer_id = c.customer_id
INNER JOIN dim_products p ON m.product_id = p.product_id
GROUP BY c.customer_name, c.segment, p.category
ORDER BY total_sales DESC
LIMIT 20;

## 57. Use CASE statements in SQL.  

In [0]:
%sql
-- Task is to apply taxes based on the segment
-- Home office gets 10%
-- Corporate 15%
-- Consumer 5%

SELECT
    segment,
CASE segment
    WHEN "Consumer" THEN 0.05
    WHEN "Home Office" THEN 0.10
    WHEN "Corporate" THEN 0.15
    ELSE 0.0
END AS Tax
FROM
    superstore_sales;
    

## 58. Create SQL views.  

In [0]:
%sql
-- Creating a view named Products
CREATE OR REPLACE VIEW Products AS
SELECT
    DISTINCT ProductID,
    ProductName,
    Category,
    SubCategory
FROM superstore_sales;

In [0]:
%sql
SELECT * FROM Products LIMIT 10;

# Phase 7 — Visualization 

## 59. Create a sales by region chart.  

In [0]:
Sales = cleaned_df.groupBy("Region") \
.agg(
    round(
        sum("Sales"),
        2
    ).alias("RegionSales")
).orderBy("Region")

px.bar(
    Sales.toPandas(),
    x = "Region",
    y = "RegionSales",
    color = "Region",
    title = "Sales by Region"
).update_layout(title_x = 0.5).show()

## 60. Create monthly sales trend chart.  

In [0]:
MonthlySales = cleaned_df.groupBy(
    month(col("OrderDate")).alias("MonthNumber"),
    date_format(col("OrderDate"), "MMMM")
    .alias("Month")
)\
.agg(
    round(
        sum("Sales"),
        2
    ).alias("MonthlySales")
).orderBy(col("MonthNumber"))

px.bar(
    MonthlySales.toPandas(),
    x = "Month",
    y = "MonthlySales",
    color = "Month",
    title = "Monthly Sales trend"
).update_layout(title_x = 0.5)

## 61. Create category-wise sales chart.  

In [0]:
CategorySales = cleaned_df.groupBy("Category") \
.agg(
    round(
        sum(col("Sales")),
        2
    ).alias("CategorySales")
).orderBy("Category")

px.bar(
    CategorySales.toPandas(),
    x = "Category",
    y = "CategorySales",
    color = "Category",
    title = "Category-wise Sales"
).update_layout(title_x = 0.5)

## 62. Create profit vs sales visualization.

In [0]:
Profit_Sales = cleaned_df.select(
    ["Sales", "Profit"]
).orderBy("Profit")

px.line(
    Profit_Sales.toPandas(),
    x = "Profit",
    y = "Sales",
    title = "Profit vs Sales"
).update_layout(title_x = 0.5)

## 63. Create top customer visualization.  

In [0]:
TopCustomer = cleaned_df.groupBy(
    col("CustomerID"),
    col("CustomerName")
).agg(
    round(
        sum(col("Sales")),
        2
    ).alias("CustomerTotalSales")
).orderBy(
    desc("CustomerTotalSales")
).limit(10)

px.bar(
    TopCustomer.toPandas(),
    x = "CustomerName",
    y = "CustomerTotalSales",
    color = "CustomerName",
    title = "Top 10 Customers by Sales"
).update_layout(title_x = 0.5)

# Phase 8 — Delta Lake & Optimization 

## 64. Save cleaned data as Delta Table.  

In [0]:
cleaned_df.write.format("delta").saveAsTable("Superstore")

## 65. Perform overwrite and append operations.  